# Notebook 02: Current Sheet Alignment and Threshold Crossing

This notebook moves from global alignment statistics to a **spatial magnetic-field model**.

Core idea:

- a current sheet creates a field-reversal layer
- cross-sheet alignment can be measured locally
- a 45° cosine threshold turns that local alignment into a spatial gate

We use the geometric condition

\[
\cos\theta \geq \frac{1}{\sqrt{1^2 + 1^2}}
\]

as a minimal precursor criterion for localized threshold structure.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 300, 300
x = np.linspace(-5.0, 5.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Synthetic current-sheet field

We use a Harris-sheet-inspired reversal in the horizontal magnetic field:

\[
B_x(y) = \tanh(y / \delta)
\]

with a small localized transverse perturbation

\[
B_y(x,y) = \epsilon \sin(kx)e^{-y^2 / \sigma^2}.
\]

This is not a full MHD solver. It is a clean geometric field model for tracking spatial alignment structure.

In [ ]:
def magnetic_field(X, Y, delta=0.5, eps=0.25, k=1.2, sigma=1.0):
    Bx = np.tanh(Y / delta)
    By = eps * np.sin(k * X) * np.exp(-(Y**2) / (sigma**2))
    return Bx, By

def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn

    # mark wrapped edges as NaN so they do not contaminate statistics
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def safe_mean(mask):
    vals = np.asarray(mask, dtype=float)
    return np.nanmean(vals)

def finite_values(arr):
    return arr[np.isfinite(arr)]

## 2. One example field

This gives a first visual for the field geometry before the thickness sweep.

In [ ]:
delta_example = 0.6
Bx, By = magnetic_field(X, Y, delta=delta_example, eps=0.25, k=1.2, sigma=1.0)
Bx_hat, By_hat = normalize_field(Bx, By)

skip = 10
plt.figure(figsize=(8, 5))
plt.quiver(X[::skip, ::skip], Y[::skip, ::skip], Bx_hat[::skip, ::skip], By_hat[::skip, ::skip])
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Normalized Magnetic Field (delta={delta_example})")
plt.show()

In [ ]:
mid_x = nx // 2

plt.figure(figsize=(8, 5))
plt.plot(y, Bx[:, mid_x], label="Bx at x=0")
plt.xlabel("y")
plt.ylabel("Bx")
plt.title("Field Reversal Across the Current Sheet")
plt.legend()
plt.show()

## 3. Cross-sheet cosine map

For each location, we compare the field just above and just below that point:

\[
\cos\theta(x,y) = \hat B(x,y+\Delta y) \cdot \hat B(x,y-\Delta y).
\]

This is the core spatial observable for the notebook.

In [ ]:
cos_map_example = cross_sheet_cosine(Bx_hat, By_hat, shift=3)

plt.figure(figsize=(8, 5))
plt.imshow(cos_map_example, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
plt.colorbar(label="cross-sheet cosine")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"Cross-Sheet Cosine Map (delta={delta_example})")
plt.show()

In [ ]:
mask_example = cos_map_example >= THRESHOLD

plt.figure(figsize=(8, 5))
plt.imshow(mask_example, extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
plt.xlabel("x")
plt.ylabel("y")
plt.title(f"45° Threshold Mask (delta={delta_example})")
plt.show()

## 4. Thickness sweep

We vary the current-sheet thickness \(\delta\). Thinner sheets create sharper reversals.

For each thickness we record:

- domain-wide threshold fraction
- threshold fraction near the sheet center
- localization ratio = center fraction / total fraction

In [ ]:
deltas = [1.2, 0.8, 0.6, 0.4, 0.25]
shift = 3
center_width = 0.45

results = []

for delta in deltas:
    Bx, By = magnetic_field(X, Y, delta=delta, eps=0.25, k=1.2, sigma=1.0)
    Bx_hat, By_hat = normalize_field(Bx, By)

    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=shift)
    mask = cos_map >= THRESHOLD

    total_fraction = np.nanmean(mask)

    center_region = np.abs(Y) < center_width
    center_fraction = np.nanmean(mask[center_region])

    localization_ratio = center_fraction / total_fraction if total_fraction > 0 else np.nan

    results.append({
        "delta": delta,
        "total_fraction": float(total_fraction),
        "center_fraction": float(center_fraction),
        "localization_ratio": float(localization_ratio),
        "cos_map": cos_map,
        "mask": mask,
        "Bx_hat": Bx_hat,
        "By_hat": By_hat,
    })

results_summary = [
    {
        "delta": r["delta"],
        "total_fraction": r["total_fraction"],
        "center_fraction": r["center_fraction"],
        "localization_ratio": r["localization_ratio"],
    }
    for r in results
]

results_summary

## 5. Cosine maps across thickness

These panels show how the spatial alignment structure changes as the sheet thins.

In [ ]:
for r in results:
    plt.figure(figsize=(8, 4.6))
    plt.imshow(r["cos_map"], extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
    plt.colorbar(label="cross-sheet cosine")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"Cross-Sheet Cosine Map (delta={r['delta']})")
    plt.show()

## 6. Threshold masks across thickness

The mask identifies where the field satisfies the 45° gate.

In [ ]:
for r in results:
    plt.figure(figsize=(8, 4.6))
    plt.imshow(r["mask"], extent=[x.min(), x.max(), y.min(), y.max()], origin="lower", aspect="auto")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"45° Threshold Mask (delta={r['delta']})")
    plt.show()

## 7. Summary curves

These summary curves are the compact output for the notebook.

In [ ]:
delta_vals = np.array([r["delta"] for r in results])
total_vals = np.array([r["total_fraction"] for r in results])
center_vals = np.array([r["center_fraction"] for r in results])
local_vals = np.array([r["localization_ratio"] for r in results])

plt.figure(figsize=(8, 5))
plt.plot(delta_vals, total_vals, marker="o", label="domain-wide threshold fraction")
plt.plot(delta_vals, center_vals, marker="s", label="center-region threshold fraction")
plt.xlabel("sheet thickness delta")
plt.ylabel("fraction passing 45°")
plt.title("Threshold Fraction vs Sheet Thickness")
plt.legend()
plt.gca().invert_xaxis()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(delta_vals, local_vals, marker="o")
plt.xlabel("sheet thickness delta")
plt.ylabel("localization ratio")
plt.title("Localization Ratio vs Sheet Thickness")
plt.gca().invert_xaxis()
plt.show()

## 8. Compact text summary

In [ ]:
for r in results_summary:
    print(
        f"delta={r['delta']:.2f} | "
        f"total_fraction={r['total_fraction']:.4f} | "
        f"center_fraction={r['center_fraction']:.4f} | "
        f"localization_ratio={r['localization_ratio']:.4f}"
    )

## 9. Interpretation

Notebook 02 supports the following minimal claim:

> A current-sheet magnetic geometry turns the 45° constraint into a localized spatial structure.  
> As the field reversal sharpens, threshold-relevant structure becomes increasingly organized near the sheet center.

This is best interpreted as a **geometric precursor condition**, not yet as a full dynamical reconnection model.

That sets up Notebook 03, where seeded fragmentation or plasmoid-like multi-layer structure can be added.